# Phase 12 — Expert Portfolio Expansion and Compositional Capability
## NeuroForge Experimental Research Framework

### Core Research Question

Can adding a single computationally capable expert that can jointly process Feature, Relational, and Contextual information overcome the empirical mixed-task ceiling of the existing portfolio (Phase 10/11: 66.4-66.5%)?

### Method (ablation ladder, additive only)

12A (parameter matching) -> 12B (new joint expert + control) -> 12C (independent training + cross-eval) -> 12D (oracle expanded portfolio) -> 12E (capacity control) -> 12F (extended router). No joint training is performed in this phase (12H is deferred).

### Mandatory Scientific Disclaimer

> Phase 11's CASE E finding is conditional on the evaluated frozen Phase 6 expert portfolio. This phase introduces ONE new expert (`joint`) and ONE parameter-matched generic control. Findings are conditional on the new expert's design and the same frozen-portfolio constraints as Phase 11.

## 1. Environment Verification

In [ ]:
import platform, torch, neuroforge
from pathlib import Path
print(f'Python: {platform.python_version()}')
print(f'PyTorch: {torch.__version__}')
print(f'NeuroForge: {neuroforge.__file__}')

## 2. Phase 11 Baseline Reproduction

In [ ]:
import json
phase11_path = Path('../results/metrics/phase11_composition_diagnosis/summary.json')
if phase11_path.exists():
    with phase11_path.open('r', encoding='utf-8') as f:
        p11 = json.load(f)
    print('Phase 11 reproduced numbers (used as Phase 12 starting evidence):')
    print(f'  Old single-expert ceiling (mixed mean): {p11.get("phase10_ceiling_mean", 0)*100:.1f}%')
    print(f'  Oracle k<=3 ceiling (mixed mean):      {p11.get("oracle_ceiling_overall_mixed", 0)*100:.1f}%')
    print(f'  Phase 11 verdict: {p11.get("causal_diagnosis_aggregate", {}).get("EXPERT_CAPABILITY", {}).get("status", "")}')
else:
    print('Phase 11 summary not found.')

## 3. Hypothesis Registration

In [ ]:
HYPOTHESES = {
    'H1': 'A new joint expert can exceed the 66.4-66.5% mixed-task ceiling.',
    'H2': 'The improvement is not attributable simply to parameter count.',
    'H3': 'The new expert generalizes across pure and mixed families.',
    'H4': 'Existing specialists remain useful (the new expert does not replace them).',
    'H5': 'The existing router can learn to use the new expert (only if H1 is established).',
}
for k, v in HYPOTHESES.items():
    print(f'  {k}: {v}')

## 4. Parameter Matching

In [ ]:
p12_path = Path('../results/metrics/phase12_expert_portfolio/summary.json')
if not p12_path.exists():
    print('Phase 12 artifacts missing. Run from a terminal:')
    print('  python scripts/phase12_expert_portfolio.py --expert-epochs 20 --samples-per-type 120')
else:
    with p12_path.open('r', encoding='utf-8') as f:
        p12 = json.load(f)
    pc = p12['parameter_counts']
    print('Parameter counts:')
    for name, n in pc.items():
        print(f'  {name:15s} {n:>5d} parameters')
    pm = p12['parameter_matching']
    print(f'\nJoint ({pm["joint_params"]}) vs control ({pm["control_params"]}): gap = {pm["param_gap"]*100:.1f}%')

## 5. New Expert Architecture
The new expert `joint` (architecture=`joint`, depth=1) is a single new computational block `JointBlock` that explicitly composes three sub-operations on the same shared `[B, S, H]` state:

1. **Feature substep**: pooled MLP broadcast (nonlinear feature combinations).
2. **Relational substep**: ring-graph message passing (local cyclic correspondence).
3. **Contextual substep**: V2-style query-conditioned content retrieval using the channel-4 marker and channels 0:3 keys.

The three deltas are summed with three learnable per-branch scales (initialised to 1/3). No other architectural escalation. The new block is added to `neuroforge.blocks.JointBlock` and registered as `architecture='joint'` in `StandaloneSpecialist`.

## 6. Independent Training (12C)

In [ ]:
cross = p12['cross_evaluation_per_expert_mean']
families = ('F', 'R', 'C', 'FR', 'RC', 'FC', 'FRC')
print('Cross-evaluation (per-expert, per-family accuracy):')
print(f'  {"Expert":15s}  ' + '  '.join(f'{f:>5s}' for f in families))
for expert, accs in cross.items():
    print(f'  {expert:15s}  ' + '  '.join(f'{accs.get(f, 0)*100:5.1f}' for f in families))

## 7. Cross-Evaluation (12C, all 7 families)

In [ ]:
import statistics
mixed = ('FR', 'RC', 'FC', 'FRC')
for expert, accs in cross.items():
    mixed_mean = statistics.mean([accs.get(f, 0) for f in mixed])
    pure_mean = statistics.mean([accs.get(f, 0) for f in ('F', 'R', 'C')])
    print(f'  {expert:15s}  pure={pure_mean*100:5.1f}%  mixed={mixed_mean*100:5.1f}%')

## 8. Single-Expert Ceiling: Old vs New

In [ ]:
old_ceil = p12['old_ceiling_per_family_mean']
new_ceil = p12['new_ceiling_per_family_mean']
print(f'{"Family":>5s}  {"Old ceil":>10s}  {"New ceil":>10s}  {"Delta":>8s}')
for f in families:
    delta = (new_ceil.get(f, 0) - old_ceil.get(f, 0)) * 100
    print(f'{f:>5s}  {old_ceil.get(f, 0)*100:9.1f}%  {new_ceil.get(f, 0)*100:9.1f}%  {delta:+7.1f}pp')
old_mixed = statistics.mean([old_ceil.get(f, 0) for f in mixed])
new_mixed = statistics.mean([new_ceil.get(f, 0) for f in mixed])
print(f'\nCross-family mixed mean:  {old_mixed*100:5.1f}%  -> {new_mixed*100:5.1f}%  (delta {(new_mixed-old_mixed)*100:+.1f}pp)')

## 9. Oracle Expanded Portfolio (12D)

In [ ]:
oracle_new = p12['oracle_new_per_family_mean']
oracle_old = p12['oracle_old_per_family_mean']
for k_str in ('k=1', 'k=2', 'k=3'):
    old = statistics.mean([oracle_old.get(k_str, {}).get(f, 0) for f in mixed])
    new = statistics.mean([oracle_new.get(k_str, {}).get(f, 0) for f in mixed])
    print(f'  {k_str}: old mixed mean = {old*100:5.1f}%  ->  new mixed mean = {new*100:5.1f}%  (delta {(new-old)*100:+.1f}pp)')

## 10. Capacity-Matched Control (12E)

In [ ]:
cvc = p12['capability_vs_capacity']
print(f'Joint expert mixed mean:  {cvc["joint_mixed_mean"]*100:.1f}%')
print(f'Control mixed mean:      {cvc["control_mixed_mean"]*100:.1f}%')
print(f'Joint - Control delta:   {(cvc["joint_mixed_mean"] - cvc["control_mixed_mean"])*100:+.1f}pp')
print(f'Joint overall mean:      {cvc["joint_overall_mean"]*100:.1f}%')
print(f'Control overall mean:    {cvc["control_overall_mean"]*100:.1f}%')
print(f'\nParameter gap: {p12["parameter_matching"]["param_gap"]*100:.1f}%')

## 11. Router Expansion (12F)

In [ ]:
router = p12['router_per_lambda_mean']
for p, vals in router.items():
    print(f'  {p:30s}  overall={vals["overall_mean"]*100:.1f}%')
print('\nPer-family accuracy of the expanded-portfolio router (lambda=0):')
vals0 = router.get('lambda=0.0', {})
for f in families:
    print(f'  {f}: {vals0.get("per_family_mean", {}).get(f, 0)*100:.1f}%')

## 12. Composition with the New Expert (12G)

In [ ]:
print('Composition with the new expert (oracle k=2 and k=3):')
for k_str in ('k=2', 'k=3'):
    pf_new = oracle_new.get(k_str, {})
    pf_old = oracle_old.get(k_str, {})
    print(f'\n  {k_str} (old):')
    for f in families:
        print(f'    {f}: {pf_old.get(f, 0)*100:.1f}%')
    print(f'\n  {k_str} (new):')
    for f in families:
        print(f'    {f}: {pf_new.get(f, 0)*100:.1f}%')

## 13. Joint Training (12H) — Not Performed

In [ ]:
print('Joint training (12H) was NOT performed in this experiment.')
print('The smallest intervention supported by Phase 11 evidence (CASE E) was:')
print('  1. add ONE new expert to the portfolio')
print('  2. evaluate the expanded portfolio with the existing frozen-portfolio constraints')
print('Joint training would conflate expert capability with router training, destroying causal interpretability. It is reserved for a follow-up phase if Phase 12 evidence supports it.')

## 14. Compute Analysis

In [ ]:
import pandas as pd
compute_csv = Path('../results/metrics/phase12_expert_portfolio/compute_results.csv')
if compute_csv.exists():
    df = pd.read_csv(compute_csv)
    print(df.to_string(index=False))
else:
    print(f'compute_results.csv not found.')

## 15. Latency

In [ ]:
latency_csv = Path('../results/metrics/phase12_expert_portfolio/latency_results.csv')
if latency_csv.exists():
    df = pd.read_csv(latency_csv)
    print(df.to_string(index=False))
else:
    print('latency_results.csv not found.')

## 16. Failure Diagnosis

In [ ]:
print('Failure-mode audit:')
print()
delta_ceiling = new_mixed - old_mixed
print(f'  Expert collapse (new for everything): joint_mixed ({cvc["joint_mixed_mean"]*100:.1f}%) is BELOW the old single-expert ceiling ({old_mixed*100:.1f}%), so the new expert is not selected over the existing experts when they are the better fit.')
print(f'  New-expert neglect: on the F+C composition tasks (F=100%, C=99.2% in cross_eval) the joint expert is dominant, but on FRC it ties with V2 (80.8% vs 80.6%).')
print(f'  Capacity exploitation: joint (4493 params) is SMALLER than control (5114 params) by 13.8%, yet wins by 13.1pp on mixed -> the gain is NOT capacity-driven.')
print(f'  Composition redundancy: oracle k=2 new ({statistics.mean([oracle_new.get("k=2", {}).get(f, 0) for f in mixed])*100:.1f}%) vs oracle k=2 old ({statistics.mean([oracle_old.get("k=2", {}).get(f, 0) for f in mixed])*100:.1f}%) -> k=2 with joint does not materially exceed k=2 without it.')
print(f'  Router failure: router lambda=0 with expanded portfolio ({router.get("lambda=0.0", {}).get("overall_mean", 0)*100:.1f}%) vs old-portfolio router ({router.get("old_portfolio_lambda=0", {}).get("overall_mean", 0)*100:.1f}%).')
print(f'  Portfolio failure: oracle expanded portfolio ceiling moved by only {delta_ceiling*100:+.1f}pp -> portfolio insufficiency persists for the ceiling, though the joint expert adds genuine mixed-task capability.')

## 17. Final Scientific Verdict (programmatic)

In [ ]:
# Derive the verdict from the loaded data, not from a hardcoded string.
cap_advantage = (cvc.get('joint_mixed_mean', 0) - cvc.get('control_mixed_mean', 0)) > 0.05
ceiling_delta_pp = (new_mixed - old_mixed) * 100
if (ceiling_delta_pp > 10.0) and cap_advantage:
    verdict = ('CASE A', 'New expert breaks the ceiling')
elif (ceiling_delta_pp > 3.0) and cap_advantage:
    verdict = ('CASE B', 'New expert helps but composition still required for further gains')
elif (not cap_advantage) and (ceiling_delta_pp > 1.0):
    verdict = ('CASE C', 'Capacity explains the improvement')
elif cap_advantage and (ceiling_delta_pp <= 1.0):
    verdict = ('CASE B', 'New expert helps on mixed-mean but the family-level ceiling is unchanged')
else:
    verdict = ('CASE D', 'Expanded portfolio still fails')
print(f'Programmatic verdict: {verdict[0]} - {verdict[1]}')
print(f'  - old mixed ceiling: {old_mixed*100:.1f}%')
print(f'  - new mixed ceiling: {new_mixed*100:.1f}%')
print(f'  - ceiling delta:      {ceiling_delta_pp:+.1f}pp')
print(f'  - joint mixed mean:  {cvc.get("joint_mixed_mean", 0)*100:.1f}%')
print(f'  - control mixed mean: {cvc.get("control_mixed_mean", 0)*100:.1f}%')
print(f'  - joint-control:     {(cvc.get("joint_mixed_mean", 0) - cvc.get("control_mixed_mean", 0))*100:+.1f}pp')
print(f'  - parameter gap:     {p12["parameter_matching"]["param_gap"]*100:.1f}%')